### **Assignment 6** — Build a Complete ML Pipeline
Objective Create a production-style ML pipeline.

In [ ]:
import numpy as np
import pandas as pd

from google.colab import files  #Comment out if you ar runnin locally

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

In [ ]:
uploaded = files.upload()  #Comment out if you ar runnin locally

Saving Housing.csv to Housing.csv


In [3]:
df = pd.read_csv('Housing.csv')
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


Handle Missing Values

In [4]:
df['total_bedrooms'] = df['total_bedrooms'].fillna(df['total_bedrooms'].mean())

In [5]:
X = df.drop('median_house_value', axis=1)
y = df['median_house_value']

In [6]:
num_cols = X.select_dtypes(include=np.number).columns
cat_cols = X.select_dtypes(exclude=np.number).columns

Preprocessing Pipeline

In [7]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(drop='first'), cat_cols)
])

Full Pipeline

In [8]:
pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', Ridge())
])

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Hyperparameter Tuning

In [10]:
params = {
    'model__alpha': [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(pipeline, params, cv=5)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_

print("Best Alpha:", grid.best_params_)

Best Alpha: {'model__alpha': 1}


In [11]:
pred = best_model.predict(X_test)

In [12]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 50709.48866938958
RMSE: 70039.80243538854
R2: 0.6256455803764769


Save Pipeline, Load & Test Saved Model

In [13]:
joblib.dump(best_model, 'full_pipeline_model.pkl')

['full_pipeline_model.pkl']

In [14]:
loaded_model = joblib.load('full_pipeline_model.pkl')

sample_pred = loaded_model.predict(X_test[:5])
print(sample_pred)

[ 64637.87138709 134789.09652688 266071.20457031 278576.34851002
 273319.71432579]
